# Validação independente — Célula 4 (Observatório do Clima)

**O que é:** refaz cada etapa **a partir dos arquivos brutos oficiais**, sem reaproveitar o código dos notebooks da célula, e compara o resultado com os CSVs que estão no repositório (`data/`).

**Como ler:** cada checagem imprime `✅` quando bate e `❌` quando diverge, sempre com o valor esperado e o obtido. Nada aqui altera os arquivos da célula.

**Arquivos necessários** (mesmos nomes que vieram das fontes):

| Fonte | Arquivo |
|---|---|
| SINISA 2024 | `SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx`, `SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx` |
| CAPAG (set/2026) | `capag-municipios-posicao-2026-set.xlsx` |
| CAPAG (a que está no Drive da célula) | `capag-municipios-2025.xlsx` |
| Atlas | `BD_Atlas_1991_2025_v1_1_2026_08_06_Consolidado.xlsx` |
| Repositório | `sinisa_limpo.csv`, `CAPAG-2025_Tratamento.csv`, `S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv`, `dataset_mestre_observatorio.csv` |

No Colab: suba todos para a pasta `/content` (ou ajuste os caminhos na célula abaixo). A leitura do Atlas leva cerca de 1 minuto.

In [1]:
import os, glob
import numpy as np
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

PASTA = os.environ.get('PASTA_DADOS', '/content')
def achar(nome):
    # procura o arquivo pelo nome (aceita prefixos no nome, ex.: 'abc123-arquivo.xlsx')
    c = sorted(glob.glob(os.path.join(PASTA, '**', '*' + nome), recursive=True))
    assert c, f'Arquivo não encontrado: {nome}'
    if len(c) > 1: print(f'(aviso) mais de um arquivo para {nome}, usando {c[0]}')
    return c[0]

ARQ = {
    'sinisa_form': achar('SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx'),
    'sinisa_ind':  achar('SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx'),
    'capag_set':   achar('capag-municipios-posicao-2026-set.xlsx'),
    'capag_drive': achar('capag-municipios-2025.xlsx'),
    'atlas':       achar('BD_Atlas_1991_2025_v1_1_2026_08_06_Consolidado.xlsx'),
    'rep_sinisa':  achar('sinisa_limpo.csv'),
    'rep_capag':   achar('CAPAG-2025_Tratamento.csv'),
    'rep_s2id':    achar('S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv'),
    'rep_mestre':  achar('dataset_mestre_observatorio.csv'),
}
for k, v in ARQ.items(): print(f'{k:12s} -> {os.path.basename(v)}')

def checa(nome, esperado, obtido):
    ok = (esperado == obtido) if not isinstance(esperado, float) else np.isclose(esperado, obtido)
    print(f"{'✅' if ok else '❌'} {nome}: esperado {esperado} | obtido {obtido}")
    return ok

sinisa_form  -> 155ed922-SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx
sinisa_ind   -> 91c4da86-SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx
capag_set    -> 81cf0ed2-capag-municipios-posicao-2026-set.xlsx
capag_drive  -> capag-municipios-2025.xlsx
atlas        -> BD_Atlas_1991_2025_v1_1_2026_08_06_Consolidado.xlsx
rep_sinisa   -> sinisa_limpo.csv
rep_capag    -> CAPAG-2025_Tratamento.csv
rep_s2id     -> S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv
rep_mestre   -> dataset_mestre_observatorio.csv


## 1. SINISA — reconstrução a partir do bruto

Lemos a aba **Dados sobre Gestão de Riscos** pela **linha de códigos** do formulário (linha 11 do Excel), e não pela posição da coluna. Os campos usados vêm do glossário oficial:
- `GAP2110` = quantidade de domicílios sujeitos a risco de inundação (declarado pelo município)
- `GAP2105` = existe mapeamento de áreas de risco de inundação?
- `GAP2103` = total de domicílios urbanos
- `Critico` = município crítico (fonte: Serviço Geológico do Brasil, campo GAP2001 do glossário)

In [2]:
bruto = pd.read_excel(ARQ['sinisa_form'], sheet_name='Dados sobre Gestão de Riscos', header=None, dtype=str)
codigos = [str(c).replace('*', '').strip() for c in bruto.iloc[10]]   # linha 11 do Excel
sin = bruto.iloc[11:].copy(); sin.columns = [c if c != 'nan' else f'col{i}' for i, c in enumerate(codigos)]
sin = sin[sin['Cod_IBGE'].str.match(r'^\d{7}$', na=False)]
sin['participa'] = sin['col12'].eq('Participante')
sin['risco'] = pd.to_numeric(sin['GAP2110'], errors='coerce')

checa('municípios no arquivo', 5570, len(sin))
checa('participantes', 4845, int(sin['participa'].sum()))
checa('declaram 0 moradias em risco', 2607, int((sin['risco'] == 0).sum()))
checa('declaram moradias em risco > 0', 2238, int((sin['risco'] > 0).sum()))
checa('total de moradias em risco no país', 2694600.0, float(sin['risco'].sum()))
print('\nMapeamento (GAP2105):', sin['GAP2105'].value_counts(dropna=False).to_dict())
print('Risco > 0 SEM mapeamento:', int(((sin['risco'] > 0) & (sin['GAP2105'] == 'Não')).sum()), 'municípios')

✅ municípios no arquivo: esperado 5570 | obtido 5570
✅ participantes: esperado 4845 | obtido 4845
✅ declaram 0 moradias em risco: esperado 2607 | obtido 2607
✅ declaram moradias em risco > 0: esperado 2238 | obtido 2238
✅ total de moradias em risco no país: esperado 2694600.0 | obtido 2694600.0

Mapeamento (GAP2105): {'Não': 3385, 'Sim': 1460, nan: 725}
Risco > 0 SEM mapeamento: 878 municípios


In [3]:
# Indicadores (arquivo 2): Crítico, IAP0002, IFD0010, IGR0001
ind_b = pd.read_excel(ARQ['sinisa_ind'], header=None, dtype=str)
cod_ind = [str(c).strip() for c in ind_b.iloc[10]]
cod_ind[:6] = ['Cod', 'Nome', 'UF', 'Regiao', 'Capital', 'Critico']
ind = ind_b.iloc[11:].copy(); ind.columns = cod_ind
ind = ind[ind['Cod'].str.match(r'^\d{7}$', na=False)]

rep = pd.read_csv(ARQ['rep_sinisa'], dtype={'Cod_IBGE_7': str}, encoding='utf-8-sig')
x = rep.merge(sin[['Cod_IBGE', 'GAP2105', 'risco']], left_on='Cod_IBGE_7', right_on='Cod_IBGE')\
       .merge(ind[['Cod', 'Critico', 'IAP0002', 'IFD0010', 'IGR0001']], left_on='Cod_IBGE_7', right_on='Cod')
num = lambda s: pd.to_numeric(s, errors='coerce').astype(float)
pares = [('qtd_domicilios_risco', 'risco', True), ('mapeamento_area_risco', 'GAP2105', False),
         ('municipio_critico', 'Critico', False), ('pct_vias_rede_drenagem', 'IAP0002', True),
         ('investimento_percapita_drenagem', 'IFD0010', True), ('pct_domicilios_risco', 'IGR0001', True)]
print('Comparação município a município: sinisa_limpo.csv (repositório) x bruto')
for a, b, eh_num in pares:
    if eh_num: dif = (~np.isclose(num(x[a]), num(x[b]), equal_nan=True)).sum()
    else:      dif = (x[a].fillna('NA') != x[b].fillna('NA')).sum()
    checa(f'{a} divergências', 0, int(dif))

Comparação município a município: sinisa_limpo.csv (repositório) x bruto
✅ qtd_domicilios_risco divergências: esperado 0 | obtido 0
✅ mapeamento_area_risco divergências: esperado 0 | obtido 0
✅ municipio_critico divergências: esperado 0 | obtido 0
✅ pct_vias_rede_drenagem divergências: esperado 0 | obtido 0
✅ investimento_percapita_drenagem divergências: esperado 0 | obtido 0
✅ pct_domicilios_risco divergências: esperado 0 | obtido 0


## 2. CAPAG — qual versão o repositório usou?

O arquivo do Tesouro tem várias abas. A nota final está na aba **Prévia da CAPAG**, coluna `CAPAG`. Comparamos o CSV do repositório com **as duas versões**: a que está no Drive da célula e a de setembro/2026.

In [4]:
def capag(arq):
    p = pd.read_excel(arq, sheet_name='Prévia da CAPAG', header=2, dtype=str)
    p = p[p['Código Município Completo'].str.match(r'^\d{7}$', na=False)]
    return p.set_index('Código Município Completo')['CAPAG'].str.strip()

def data_ranking(arq):
    r = pd.read_excel(arq, sheet_name='Ranking', header=None, nrows=4, dtype=str)
    return [v for v in r.values.ravel() if isinstance(v, str) and ('Data' in v or '2026' in v)][:4]

cap_set, cap_drive = capag(ARQ['capag_set']), capag(ARQ['capag_drive'])
print('Datas na aba Ranking (Drive):', data_ranking(ARQ['capag_drive']))
print('Datas na aba Ranking (set/26):', data_ranking(ARQ['capag_set']))
print('\nDistribuição Drive :', cap_drive.value_counts().to_dict())
print('Distribuição set/26:', cap_set.value_counts().to_dict())

rc = pd.read_csv(ARQ['rep_capag'], dtype=str, encoding='utf-8-sig').set_index('cod_ibge')['capag']
checa('repositório x CAPAG do Drive (divergências)', 0, int((rc != cap_drive.reindex(rc.index)).sum()))
d = (rc != cap_set.reindex(rc.index))
checa('repositório x CAPAG set/26 (divergências)', 0, int(d.sum()))
print('\nExemplos de municípios que mudaram de nota:')
ex = pd.DataFrame({'repositorio': rc[d], 'set_2026': cap_set.reindex(rc.index)[d]})
ex = ex.join(sin.set_index('Cod_IBGE')['Nome_Mun'])
print(ex.loc[[c for c in ['3303906', '3300100', '4202404', '3156700'] if c in ex.index]])
print(pd.crosstab(ex['repositorio'], ex['set_2026']))

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Datas na aba Ranking (Drive): ['Data', 'Data do Ranking: 12/04', '2026-04-13 00:00:00', 'Data de Processamento do Ranking: 13/04']
Datas na aba Ranking (set/26): ['Data', 'Data de atualização', '2026-08-31 00:00:00', 'Data do ranking']

Distribuição Drive : {'C': 2184, 'A': 1190, 'B': 976, 'n.d.': 683, 'A+': 265, 'n.e.': 137, 'B+': 113, 'D': 19}
Distribuição set/26: {'C': 2274, 'A': 1096, 'B': 963, 'n.d.': 426, 'A+': 418, 'B+': 200, 'n.e.': 170, 'D': 20}
✅ repositório x CAPAG do Drive (divergências): esperado 0 | obtido 0
❌ repositório x CAPAG set/26 (divergências): esperado 0 | obtido 867

Exemplos de municípios que mudaram de nota:
         repositorio set_2026        Nome_Mun
cod_ibge                                     
3303906         n.d.        A      Petrópolis
3300100         n.d.        C  Angra dos Reis
4202404            B        C        Blumenau
3156700            B        C          Sabará
set_2026      A   A+   B   B+    C  D  n.d.  n.e.
repositorio                     

## 3. Atlas — recorte e danos humanos

Recorte usado pela célula: grupo **Hidrológico** (classificação da própria aba *Grupos de Desastres* do Atlas), eventos de 2021 a 2025, Brasil inteiro, aba **Atlas Valores Corrigidos**.

In [5]:
cols = ['Protocolo_S2iD', 'Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'Cod_Cobrade', 'descricao_tipologia',
        'grupo_de_desastre', 'Cod_IBGE_Mun', 'Status', 'DH_MORTOS', 'DH_FERIDOS', 'DH_ENFERMOS', 'DH_DESABRIGADOS',
        'DH_DESALOJADOS', 'DH_DESAPARECIDOS', 'DH_total_danos_humanos_diretos', 'DM_total_danos_materiais', 'PEPL_total_publico']
atlas = pd.read_excel(ARQ['atlas'], sheet_name='Atlas Valores Corrigidos', usecols=cols, dtype=str)
atlas['data'] = pd.to_datetime(atlas['Data_Evento'], errors='coerce')
rec = atlas[(atlas['grupo_de_desastre'] == 'Hidrológico') & atlas['data'].dt.year.between(2021, 2025)].copy()
for c in cols[9:]: rec[c] = pd.to_numeric(rec[c], errors='coerce').fillna(0)

s2 = pd.read_csv(ARQ['rep_s2id'], sep=';', dtype={'Cod_IBGE_Mun': str}, encoding='utf-8-sig')
checa('registros no recorte', 9890, len(rec))
checa('mesmos protocolos no repositório', 9890, int(rec['Protocolo_S2iD'].isin(s2['Protocolo_S2iD']).sum()))
m = s2.merge(rec, on='Protocolo_S2iD', suffixes=('_rep', ''))
_ = checa('danos materiais (diferença > R$ 0,01)', 0, int((abs(m['DM_total_danos_materiais_rep'] - m['DM_total_danos_materiais']) > 0.01).sum()))

✅ registros no recorte: esperado 9890 | obtido 9890
✅ mesmos protocolos no repositório: esperado 9890 | obtido 9890
✅ danos materiais (diferença > R$ 0,01): esperado 0 | obtido 0


In [6]:
seis = ['DH_MORTOS', 'DH_FERIDOS', 'DH_ENFERMOS', 'DH_DESABRIGADOS', 'DH_DESALOJADOS', 'DH_DESAPARECIDOS']
soma6 = m[seis].sum(axis=1)
checa('total oficial == soma das 6 categorias (registros que NÃO batem)', 0, int((soma6 != m['DH_total_danos_humanos_diretos']).sum()))
sem_feridos = m[[c for c in seis if c != 'DH_FERIDOS']].sum(axis=1)
checa('DH_total_calculado do repositório == soma SEM feridos (registros que NÃO batem)', 0, int((sem_feridos != m['DH_total_calculado']).sum()))
checa('registros com feridos > 0 (= as "606 inconsistências")', 606, int((m['DH_FERIDOS'] > 0).sum()))
print('\nExemplo:')
print(m.loc[m['DH_FERIDOS'] > 0, ['Nome_Municipio_rep', 'DH_FERIDOS', 'DH_total_calculado', 'DH_total_danos_humanos_diretos']].head(3).to_string())

✅ total oficial == soma das 6 categorias (registros que NÃO batem): esperado 0 | obtido 0
✅ DH_total_calculado do repositório == soma SEM feridos (registros que NÃO batem): esperado 0 | obtido 0
✅ registros com feridos > 0 (= as "606 inconsistências"): esperado 606 | obtido 606

Exemplo:
   Nome_Municipio_rep  DH_FERIDOS  DH_total_calculado  DH_total_danos_humanos_diretos
3             Ubatuba           1                 532                             533
19          Três Rios          30                 402                             432
24          Barbacena           1                 113                             114


## 4. Cruzamento — totais de controle do `dataset_mestre_observatorio.csv`

In [7]:
mestre = pd.read_csv(ARQ['rep_mestre'], dtype={'Cod_IBGE_7': str}, encoding='utf-8-sig')
checa('linhas (1 por município)', 5570, len(mestre))
checa('código IBGE duplicado', 0, int(mestre['Cod_IBGE_7'].duplicated().sum()))
checa('soma de eventos = registros do recorte', 9890, int(mestre['total_desastres'].sum()))
checa('soma de danos materiais = soma do recorte (R$)', round(rec['DM_total_danos_materiais'].sum(), 0), round(mestre['soma_danos_materiais'].sum(), 0))
checa('soma_afetados = total oficial de danos humanos', int(rec['DH_total_danos_humanos_diretos'].sum()), int(mestre['soma_afetados'].sum()))
checa('municípios sem nota CAPAG no mestre', 0, int(mestre['capag'].isna().sum()))
print('   ->', mestre.loc[mestre['capag'].isna(), ['Nome', 'UF']].values.tolist())

✅ linhas (1 por município): esperado 5570 | obtido 5570
✅ código IBGE duplicado: esperado 0 | obtido 0
✅ soma de eventos = registros do recorte: esperado 9890 | obtido 9890
✅ soma de danos materiais = soma do recorte (R$): esperado 43291388563.0 | obtido 43291388564.0
❌ soma_afetados = total oficial de danos humanos: esperado 3525510 | obtido 3497101
❌ municípios sem nota CAPAG no mestre: esperado 0 | obtido 3
   -> [['Calçoene', 'AP'], ['Fernando de Noronha', 'PE'], ['Brasília', 'DF']]


## 5. Funil — reproduzindo o notebook `Transformacao-Filtros` e a versão corrigida

O filtro 4 do notebook (célula 4, linha 18) usa `.notna()`. Como a coluna tem `'Sim'`/`'Não'`, esse filtro não remove ninguém. Abaixo, as duas versões lado a lado, com as duas CAPAGs.

In [8]:
base = sin[['Cod_IBGE', 'Nome_Mun', 'UF', 'participa', 'risco', 'GAP2105']].rename(columns={'Cod_IBGE': 'cod', 'Nome_Mun': 'nome', 'GAP2105': 'mapeado'})
base = base.merge(ind[['Cod', 'Critico']], left_on='cod', right_on='Cod')
ag = rec.groupby('Cod_IBGE_Mun').agg(eventos=('Protocolo_S2iD', 'count'))
base = base.join(ag, on='cod').fillna({'eventos': 0})
base['capag_drive'] = base['cod'].map(cap_drive); base['capag_set'] = base['cod'].map(cap_set)

def funil(capcol, filtro_mapa):
    f1 = base[base['participa']]; f2 = f1[f1['risco'] > 0]
    f3 = f2[f2[capcol].isin(['C', 'D'])]
    f4 = f3[f3['mapeado'].notna()] if filtro_mapa == 'notna (notebook)' else f3[f3['mapeado'] == 'Sim']
    top = f4.sort_values(['risco', 'eventos'], ascending=False).head(20)
    return [len(f1), len(f2), len(f3), len(f4)], top

res = {}
for capcol in ['capag_drive', 'capag_set']:
    for fm in ['notna (notebook)', "== 'Sim' (corrigido)"]:
        n, top = funil(capcol, fm); res[(capcol, fm)] = top
        print(f'{capcol:12s} | filtro4 {fm:22s} | funil {n} | moradias top20 = {int(top.risco.sum()):,}')
n, _ = funil('capag_drive', 'notna (notebook)')
_ = checa('reproduz o notebook (universo = 816)', 816, n[3])

capag_drive  | filtro4 notna (notebook)       | funil [4845, 2238, 816, 816] | moradias top20 = 684,758
capag_drive  | filtro4 == 'Sim' (corrigido)   | funil [4845, 2238, 816, 499] | moradias top20 = 668,401
capag_set    | filtro4 notna (notebook)       | funil [4845, 2238, 850, 850] | moradias top20 = 715,822
capag_set    | filtro4 == 'Sim' (corrigido)   | funil [4845, 2238, 850, 526] | moradias top20 = 701,607
✅ reproduz o notebook (universo = 816): esperado 816 | obtido 816


In [9]:
a = res[('capag_drive', 'notna (notebook)')]; b = res[('capag_drive', "== 'Sim' (corrigido)")]; c = res[('capag_set', "== 'Sim' (corrigido)")]
print('Saem do top 20 ao corrigir o filtro 4:', a.loc[~a.cod.isin(b.cod), ['nome', 'UF', 'mapeado']].values.tolist())
print('Saem ao trocar para a CAPAG set/26  :', b.loc[~b.cod.isin(c.cod), ['nome', 'UF']].values.tolist())
print('Entram com a CAPAG set/26           :', c.loc[~c.cod.isin(b.cod), ['nome', 'UF', 'capag_drive', 'capag_set']].values.tolist())
print('\nTop 20 (filtro corrigido + CAPAG set/26):')
print(c[['nome', 'UF', 'risco', 'capag_set', 'Critico', 'eventos']].reset_index(drop=True).to_string())
_ = checa('todos do top 20 são "Críticos"', 20, int((c['Critico'] == 'Sim').sum()))

Saem do top 20 ao corrigir o filtro 4: [['São João de Meriti', 'RJ', 'Não'], ['Cametá', 'PA', 'Não']]
Saem ao trocar para a CAPAG set/26  : [['Palmares', 'PE'], ['Barreiros', 'PE'], ['Paraty', 'RJ']]
Entram com a CAPAG set/26           : [['Blumenau', 'SC', 'B', 'C'], ['Sabará', 'MG', 'B', 'C'], ['Angra dos Reis', 'RJ', 'n.d.', 'C']]

Top 20 (filtro corrigido + CAPAG set/26):
                     nome  UF     risco capag_set Critico  eventos
0             São Gonçalo  RJ  175838.0         C     Sim      4.0
1            Campo Grande  MS  125000.0         C     Sim     56.0
2                  Canoas  RS   70000.0         C     Sim      7.0
3   Campos dos Goytacazes  RJ   63000.0         C     Sim      8.0
4         Duque de Caxias  RJ   31070.0         C     Sim      6.0
5                  Esteio  RS   30000.0         C     Sim      5.0
6                Blumenau  SC   28758.0         C     Sim      7.0
7                São José  SC   25000.0         C     Sim      2.0
8                 